In [1]:
# Cell 1: Import and load all raw data saved in data collection notebook (notebook 01)
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

# Load saved data
uk_cpi = pd.read_csv('../data/raw/uk_cpi.csv', index_col=0, parse_dates=True)
uk_wages = pd.read_csv('../data/raw/uk_wages.csv', index_col=0, parse_dates=True)
uk_unemp = pd.read_csv('../data/raw/uk_unemployment.csv', index_col=0, parse_dates=True)
oil = pd.read_csv('../data/raw/brent_crude.csv', index_col=0, parse_dates=True)
gbp_usd = pd.read_csv('../data/raw/gbp_usd.csv', index_col=0, parse_dates=True)
us_cpi = pd.read_csv('../data/raw/us_cpi.csv', index_col=0, parse_dates=True)

print("All datasets loaded :)")
print(f"CPI shape: {uk_cpi.shape}")
print(f"Wages shape: {uk_wages.shape}")
print(f"Unemployment shape: {uk_unemp.shape}")
print(f"Oil shape: {oil.shape}")
print(f"GBP/USD shape: {gbp_usd.shape}")
print(f"US CPI shape: {us_cpi.shape}")

All datasets loaded :)
CPI shape: (452, 1)
Wages shape: (307, 1)
Unemployment shape: (318, 1)
Oil shape: (6988, 1)
GBP/USD shape: (6990, 1)
US CPI shape: (321, 1)


In [2]:
# Cell 2: Convert daily data to monthly averages
# 'MS' means Month Start - so every date will be the 1st of the month, making it easier to merge with monthly CPI data later

oil_monthly = oil.resample('MS').mean()
gbp_monthly = gbp_usd.resample('MS').mean()

# Make sure US CPI is on Month Start, even though it's already monthly
us_cpi_monthly = us_cpi.resample('MS').mean()

print("Daily → Monthly conversion done :)")
print(f"Oil monthly shape: {oil_monthly.shape}")
print(f"GBP/USD monthly shape: {gbp_monthly.shape}")
print(f"\nLatest oil monthly average: {oil_monthly.iloc[-1].values[0]:.2f} USD/barrel")
print(f"Latest GBP/USD monthly average: {gbp_monthly.iloc[-1].values[0]:.4f}")

Daily → Monthly conversion done :)
Oil monthly shape: (322, 1)
GBP/USD monthly shape: (322, 1)

Latest oil monthly average: 126.98 USD/barrel
Latest GBP/USD monthly average: 1.3234


In [3]:
# Drop the current incomplete month from daily-converted series as the average would be misleading
oil_monthly = oil_monthly.iloc[:-1]
gbp_monthly = gbp_monthly.iloc[:-1]

print("Latest complete month oil:", oil_monthly.index[-1].date(), 
      f"= ${oil_monthly.iloc[-1].values[0]:.2f}/barrel")
print("Latest complete month GBP/USD:", gbp_monthly.index[-1].date(), 
      f"= {gbp_monthly.iloc[-1].values[0]:.4f}")

# Merge everything into one dataframe
df = uk_cpi.copy()
df = df.join(uk_wages, how='left')
df = df.join(uk_unemp, how='left')
df = df.join(oil_monthly, how='left')
df = df.join(gbp_monthly, how='left')
df = df.join(us_cpi_monthly, how='left')

# Sanity check data frame
print(f"\nMerged dataframe shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDate range: {df.index[0].date()} to {df.index[-1].date()}")
print("\nMost recent figures:\n")
print(df.tail())

Latest complete month oil: 2026-09-01 = $114.16/barrel
Latest complete month GBP/USD: 2026-09-01 = 1.3406

Merged dataframe shape: (452, 6)
Columns: ['cpi', 'wages', 'unemployment', 'oil_price', 'gbp_usd', 'us_cpi']

Date range: 1989-01-01 to 2026-08-01

Most recent figures:

            cpi     wages  unemployment   oil_price   gbp_usd   us_cpi
date                                                                  
2026-04-01  2.8  4.861151           4.9  117.287500  1.345405  332.407
2026-05-01  2.8  2.519524           4.9  107.139474  1.349835  333.979
2026-06-01  2.6  2.542015           4.9   85.399091  1.333776  332.568
2026-07-01  2.9  2.411324           NaN   83.758696  1.338341  332.813
2026-08-01  3.1       NaN           NaN   91.076000  1.354690  334.131


In [4]:
# Cell 4: Check missing values across the whole dataframe
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nTotal rows: {len(df)}")

# Find the first date where ALL columns have data
first_complete = df.dropna().index[0]
print(f"\nFirst date with complete data across all columns: {first_complete.date()}")

# Trim the dataframe to start from that date
df = df[df.index >= first_complete]

print(f"\nTrimmed dataframe shape: {df.shape}")
print(f"New date range: {df.index[0].date()} to {df.index[-1].date()}")
print("\nMissing values after trimming:")
print(df.isnull().sum())

Missing values per column:
cpi               0
wages           145
unemployment    134
oil_price       132
gbp_usd         132
us_cpi          133
dtype: int64

Total rows: 452

First date with complete data across all columns: 2001-01-01

Trimmed dataframe shape: (308, 6)
New date range: 2001-01-01 to 2026-08-01

Missing values after trimming:
cpi             0
wages           1
unemployment    2
oil_price       0
gbp_usd         0
us_cpi          1
dtype: int64


In [5]:
# Cell 5: Fill remaining missing values (updated syntax for newer pandas)
df = df.ffill()  # forward fill
df = df.bfill()  # backward fill

print("Missing values after filling:")
print(df.isnull().sum())

# Cell 6: Create lag variables to better capture variable relationships
df['oil_lag1'] = df['oil_price'].shift(1)
df['oil_lag2'] = df['oil_price'].shift(2)
df['oil_lag3'] = df['oil_price'].shift(3)
df['gbp_lag1'] = df['gbp_usd'].shift(1)
df['gbp_lag2'] = df['gbp_usd'].shift(2)
df['wages_lag1'] = df['wages'].shift(1)
df['wages_lag3'] = df['wages'].shift(3)

# Drop the first 3 rows which will have NaN from the lagging
df = df.dropna()

print(f"\nFinal dataframe shape: {df.shape}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\nAll columns: {df.columns.tolist()}")

Missing values after filling:
cpi             0
wages           0
unemployment    0
oil_price       0
gbp_usd         0
us_cpi          0
dtype: int64

Final dataframe shape: (305, 13)
Date range: 2001-04-01 to 2026-08-01

All columns: ['cpi', 'wages', 'unemployment', 'oil_price', 'gbp_usd', 'us_cpi', 'oil_lag1', 'oil_lag2', 'oil_lag3', 'gbp_lag1', 'gbp_lag2', 'wages_lag1', 'wages_lag3']


In [6]:
# Cell 7: Save the clean merged dataset
df.to_csv('../data/processed/merged_clean.csv')

print("Clean dataset saved to data/processed/merged_clean.csv :)")
print("\n=== CLEANING COMPLETE ===")
print(f"Observations: {len(df)}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Features: {df.columns.tolist()}")
print("\nFinal snapshot:")
print(df.tail())

Clean dataset saved to data/processed/merged_clean.csv :)

=== CLEANING COMPLETE ===
Observations: 305
Date range: 2001-04-01 to 2026-08-01
Features: ['cpi', 'wages', 'unemployment', 'oil_price', 'gbp_usd', 'us_cpi', 'oil_lag1', 'oil_lag2', 'oil_lag3', 'gbp_lag1', 'gbp_lag2', 'wages_lag1', 'wages_lag3']

Final snapshot:
            cpi     wages  unemployment   oil_price   gbp_usd   us_cpi  \
date                                                                     
2026-04-01  2.8  4.861151           4.9  117.287500  1.345405  332.407   
2026-05-01  2.8  2.519524           4.9  107.139474  1.349835  333.979   
2026-06-01  2.6  2.542015           4.9   85.399091  1.333776  332.568   
2026-07-01  2.9  2.411324           4.9   83.758696  1.338341  332.813   
2026-08-01  3.1  2.411324           4.9   91.076000  1.354690  334.131   

              oil_lag1    oil_lag2    oil_lag3  gbp_lag1  gbp_lag2  \
date                                                                 
2026-04-01  103.134